### 문항 1 아실 아파트 목록, 매물 크롤링

In [ ]:
# 요구사항 1. 아실 사이트에서 수집을 두 단계로 분리해 작성하시오.

# 대상: https://asil.kr/asil/index.jsp

# 1단계 collect_apt.py — 행정동 별로 아파트 목록을 모아 apts.csv로 저장

# 2단계 collect_forsale.py — apts.csv를 읽어 매물 페이지를 수집해 forsales.csv로 저장 (status를 최신화, done, failed)

# apts.csv는 다음 컬럼을 가질 것: seq, status, collected_at +a (상세칼럼들)

# status의 초기값은 pending

# 요청 간 0.5초 이상 지연

# 요구사항 2. 재시도 & 로깅 & 실패 큐 처리 로직 적용

# 재시도 — 지수 백오프. 재시도 대상은 Timeout·ConnectionError·429·5xx로 한정하고, 404·400은 즉시 포기할 것

# 로깅 — print 대신 logging. 파일과 화면에 동시 출력, INFO/WARNING/ERROR 구분

# 실패 큐 — 실패한 seq을 오류 유형과 함께 failed_apt.csv, failed_forsale.csv 각각 분리하여 저장

# 통계 — 종료 시 성공 / 실패 / 건너뜀 건수를 한 줄로 출력

# 검증 — 큐의 URL 중 일부를 존재하지 않는 주소로 바꿔 넣고, 크롤러가 죽지 않고 끝까지 도는지 확인할 것

In [24]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import unquote
from datetime import datetime
import pandas as pd

In [18]:
URL = 'https://asil.kr/app/data/data_apt_list.jsp'

param_str ='dong=1174011000&building=&household=50&order=0&order_type=0'
params =  [p.split('=') for p in param_str.split('&')]
params = {k: v for k, v in params}
PARAMS = {'dong': '1174011000',
 'building': 'apt',
 'household': '50',
 'order': '0',
 'order_type': '0'}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://asil.kr/app/apt_list.jsp'
}


In [4]:
dong_code ={
    '서울특별시 강동구 명일동' : '1174010100',
    '서울특별시 강동구 고덕동' : '1174010200',
    '서울특별시 강동구 상일동' : '1174010300',
    '서울특별시 강동구 길동'   : '1174010500',
	'서울특별시 강동구 둔촌동' : '1174010600',
	'서울특별시 강동구 암사동' : '1174010700',
	'서울특별시 강동구 성내동' : '1174010800',
	'서울특별시 강동구 천호동' : '1174010900',
	'서울특별시 강동구 강일동' : '1174011000',
}

In [20]:
res = requests.get(URL, params={**PARAMS}, headers=HEADERS)
res.raise_for_status
res.json()[0]

{'building': 'apt',
 'seq': '20412014',
 'name': '강동리버스트4단지',
 'dong': '1174011000',
 'dongname': '강일동',
 'bungi': '114',
 'movein': '2020',
 'household': '1,239',
 'total_dong': '10',
 'type': '0',
 'etc': '',
 'offer': '매물 21건',
 'lat': '37.572919806',
 'lng': '127.17222504'}

In [34]:
### 최적화(아파트 목록 수집)
URL = 'https://asil.kr/app/data/data_apt_list.jsp'
PARAMS = {
 'building': 'apt',
 'household': '50',
 'order': '0',
 'order_type': '0'}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://asil.kr/app/apt_list.jsp'
}

dong_code ={
    '서울특별시 강동구 명일동' : '1174010100',
    '서울특별시 강동구 고덕동' : '1174010200',
    '서울특별시 강동구 상일동' : '1174010300',
    '서울특별시 강동구 길동'   : '1174010500',
	'서울특별시 강동구 둔촌동' : '1174010600',
	'서울특별시 강동구 암사동' : '1174010700',
	'서울특별시 강동구 성내동' : '1174010800',
	'서울특별시 강동구 천호동' : '1174010900',
	'서울특별시 강동구 강일동' : '1174011000',
}

def fetch(dong):
    res = res = requests.get(URL, params={**PARAMS, 'dong':dong}, headers=HEADERS)
    res.raise_for_status()
    return res.json()

def parse(datas):
    apt_list = []
    for data in datas:
        apt_list.append({
            'seq': data.get('seq',''),
            '동': data.get('dongname', ''),
            '단지명': data.get('name',''),
            '세대수': data.get('household',''),
            '건축년도': data.get('movein', ''),
            '매물수': data.get('offer',''),
            '위도': data.get('lat',''),
            '경도': data.get('lng',''),
        })
    return apt_list

result = []
for dong in dong_code.values():
    result.extend(parse(fetch(dong)))

df = pd.DataFrame(result).to_csv('apts.csv', index=False, encoding='utf-8-sig')

In [115]:
URL = 'https://realty.asil.kr/api_asil/data_sale_of_apt_nomal.aspx'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://asil.kr'
}

data_str = 'asil_bldcode=1532&focus_bldcode=1532&oidx=1&oby=down&total=20&dealmode=&dong=&user=&ptp_no=&asil_preminum=-19622%2C-18881&asil_focus=&last_mm_num=0&pre_mm_uid=41293668%2C40880165%2C40836774%2C41293672&focus_mm_uid='
data = [p.split('=') for p in data_str.split('&')]
data = {k:v for k,v in data}
DATA = {'asil_bldcode': '1532',
 'focus_bldcode': '1532',
 'oidx': 1,
 'oby': 'down',
 'total': 20,
 'last_mm_num': 0,
 }

In [ ]:
res = requests.post(URL, data={**DATA}, headers=HEADERS)
res.raise_for_status()
res.json()['list_result'][0]

{'mm_uid': '41397235',
 'RLSTTYPE_CD': 'A01',
 'RLSTTYPE_NM': '아파트',
 'MAP_X': '127.151556',
 'MAP_Y': '37.547941',
 'BDONG_NM': '905',
 'SPLY_SPC': '104.30',
 'EXCLS_SPC': '83.93',
 'TOT_FLR_CNT': '15',
 'CORES_FLR_CNT': '8',
 'DEALTYPE_NM': '전세',
 'FETR_DESC': '융무. 남향. 뻥뷰.  수리된 집. 샷시 해드림.',
 'DEAL_AMT': '200,000',
 'WRRNT_AMT': '75,000',
 'LEASE_AMT': '0',
 'BLDNM': '고덕주공9단지',
 'DEALTYPE_CD': 'B01',
 'SUB_RLSTTYPE_NM': '아파트',
 'CORES_FLR_CNT_NM': '중',
 'TOT_CNT': '33',
 'PHTO_PATH': '',
 'MM_IDX': '1',
 'next_flag': False,
 'SVC_DATE_STRT': '2026-08-24',
 'BRKG_NM': '뉴욕부동산중개(주)',
 'MAP_LOC_YN': ' ',
 'premium_price': '0',
 'prcl_price': '0',
 'grnd_spc': '0.00',
 'TOT_SPC': '0.00',
 'SUB_RLSTTYPE_CD': 'A01',
 'CTRT_SPC': '104.30',
 'CNST_SPC': '0.00',
 'spc_v1': '104.30',
 'spc_v2': '83.93',
 'spc_py_v1': '32',
 'spc_py_v2': '25',
 'PHTO_CNT': '0',
 'PRTN_IMG': '',
 'user_id': '-25567',
 'f_option': '0',
 'FLR_DP_MTHD_CD': '2',
 'RPRST_TEL': '02-481-3003',
 'TEL_ADD': '010-5701-2003'

In [ ]:
URL = 'https://realty.asil.kr/api_asil/data_sale_of_apt_nomal.aspx'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://asil.kr'
}

DATA = {
 'oidx': 1,
 'oby': 'down',
 'total': 20,
 'last_mm_num': 0,
 }

def fetch(seq):
    res = requests.post(URL, data={**DATA, 'asil_bldcode': seq, 'focus_bldcode': seq }, headers=HEADERS)
    res.raise_for_status()
    return res.json()

def parse(datas, seq):
    detail = []
    for data in datas['list_result']:
        detail.append({
            'seq': seq,
            'uid': data.get('mm_uid', ''),
            '상세': data.get('FETR_DESC', ''),
            '중개사': data.get('BRKG_NM', ''),
            '매물유형': data.get('DEALTYPE_NM', ''),
            '동': data.get('BDONG_NM', ''),
            '층': data.get('CORES_FLR_CNT_NM', ''),
            '공급면적': data.get('CTRT_SPC', '') if data.get('CTRT_SPC') else data.get('SPLY_SPC', ''),
            '전용면적': data.get('EXCLS_SPC', ''),
            '매매가': data.get('DEAL_AMT', ''),
            '보증금': data.get('WRRNT_AMT', ''),
            '월세': data.get('LEASE_AMT', ''),
            '등록일': data.get('SVC_DATE_STRT', ''),
        })

    return detail